In [1]:
from tqdm import tqdm
import pandapower as pp
import pandas as pd
import numpy as np
from pandapower.pypower.makePTDF import makePTDF
import pandapower.converter as pc
import pyomo.environ as pyo
from scipy.sparse import lil_matrix
from matpowercaseframes import CaseFrames
import pandas as pd
from scipy.stats import multivariate_normal
import pyscipopt
from numba import njit
from pyscipopt import quicksum

In [306]:
%%time
mpc_file = 'julia\\pglib-opf-17.08\\pglib_opf_case3_lmbd.m'
mpc_frames = CaseFrames(mpc_file)
model = pyscipopt.Model()
ngen = len(mpc_frames.gen)
gen_lb = mpc_frames.gen.PMIN.values
gen_ub = mpc_frames.gen.PMAX.values
def variable_p(ngen):
    p = np.zeros(ngen, dtype=object)
    p_init = mpc_frames.gen.PG.values
    p_init_sol = model.createPartialSol()
    for g in range(len(p)):
        p[g] = model.addVar(vtype='C', lb=gen_lb[g], ub=gen_ub[g], name=f'p_{g}') # decision variables p
        model.setSolVal(p_init_sol, p[g], p_init[g]) # initial value of variables p
    return p
p = variable_p(ngen)
# objective (1a)  
cost_coeff = mpc_frames.gencost.loc[:,'COST_2':'COST_0'].values
# data for constraint (1b) μ is omitted
sigma_scaling = 0.03
d = mpc_frames.bus.PD.values
nonzeroindices = np.nonzero(d)[0]
d_nonzeros = d[d!=0]
sigma = (sigma_scaling * d_nonzeros)
mean = np.zeros(len(d_nonzeros))
covariance_matrix = np.diag(sigma)**2
omega_dist = multivariate_normal(mean=mean, cov=covariance_matrix)
num_samples = 1
omega_samples = omega_dist.rvs(size=num_samples)
nbus = len(mpc_frames.bus)
omega = np.zeros((num_samples,nbus))
omega[:,nonzeroindices] = omega_samples
omega = omega[0]
# constraint (1b)
c1b = model.addCons(quicksum([p[g] for g in range(ngen)]) ==\
                    d.sum() - omega.sum(), name='c1b')
# constraint (1c) contained in variable p
# constraint (1d) μ is omitted
ppc = {key: mpc_frames.__getattribute__(key) if not isinstance(
    mpc_frames.__getattribute__(key), pd.DataFrame) else mpc_frames.__getattribute__(
    key).values for key in mpc_frames._attributes}
baseMVA, bus, branch, gen = ppc['baseMVA'], ppc['bus'], ppc['branch'], ppc['gen']
bus_index_dict = {}
for i, index in enumerate(ppc['bus'][:,0]):
    bus_index_dict[index.astype(int)] = i
for i,bus_name in enumerate(bus[:,0]):
    bus[i,0] = bus_index_dict[bus_name]
for i,fbus_name in enumerate(branch[:,0]):
    branch[i,0] = bus_index_dict[fbus_name]
for i,tbus_name in enumerate(branch[:,1]):
    branch[i,1] = bus_index_dict[tbus_name]
for i,gen_name in enumerate(gen[:,0]):
    gen[i,0] = bus_index_dict[gen_name]
M = makePTDF(baseMVA, bus, branch)
bus_index = bus[:,0].astype(int)
gen_index = gen[:,0].astype(int)
H = np.zeros((len(bus_index), len(gen_index)), dtype=int)
for col, gen_bus in enumerate(gen_index):
    H[gen_bus, col] = 1
f_max = mpc_frames.branch.RATE_A.values
nline = len(mpc_frames.branch)
c1d_lb = np.zeros(nline, dtype=object)
c1d_ub = np.zeros(nline, dtype=object)
def flow_expr():
    for l in range(nline):
        c1d_lb[l] = model.addCons(quicksum(M[l,b]*quicksum(H[b,g]*p[g] for g in range(ngen))\
                                 +omega[b] - d[b] for b in range(nbus)) >= -f_max[l], name=f'c1d_lb_{l}')
        c1d_ub[l] = model.addCons(quicksum(M[l,b]*quicksum(H[b,g]*p[g] for g in range(ngen))\
                                 +omega[b] - d[b] for b in range(nbus)) <= f_max[l], name=f'c1d_ub_{l}')
flow_expr()
model.setObjective(quicksum([cost_coeff[:,0][g]*p[g] for g in range(ngen)])\
                 + quicksum([cost_coeff[:,1][g]*p[g] for g in range(ngen)])\
                 + cost_coeff[:,2].sum(), sense='minimize')
model.optimize()

CPU times: total: 31.2 ms
Wall time: 35.3 ms


In [78]:
from pyscipopt import Model, quicksum, multidict

def eoq(I, F, h, d, w, W, a0, aK, K):
    """eoq --  multi-item capacitated economic ordering quantity model
    Parameters:
        - I: set of items
        - F[i]: ordering cost for item i
        - h[i]: holding cost for item i
        - d[i]: demand for item i
        - w[i]: unit weight for item i
        - W: capacity (limit on order quantity)
        - a0: lower bound on the cycle time (x axis)
        - aK: upper bound on the cycle time (x axis)
        - K: number of linear pieces to use in the approximation
    Returns a model, ready to be solved.
    """
    # construct points for piecewise-linear relation, store in a,b
    a, b = {}, {}
    delta = float(aK - a0) / K
    for i in I:
        for k in range(K):
            T = a0 + delta * k
            a[i, k] = T  # abscissa: cycle time
            b[i, k] = F[i] / T + h[i] * d[i] * T / 2.  # ordinate: (convex) cost for this cycle time
    
    model = Model("multi-item, capacitated EOQ")
    model.setPresolve(SCIP_PARAMSETTING.OFF)
    model.setHeuristics(SCIP_PARAMSETTING.OFF)
    model.disablePropagation()
    x, c, w_ = {}, {}, {}
    for i in I:
        x[i] = model.addVar(vtype="C", name="x(%s)" % i)  # cycle time for item i
        c[i] = model.addVar(vtype="C", name="c(%s)" % i)  # total cost for item i
        for k in range(K):
            w_[i, k] = model.addVar(ub=1, vtype="C", name="w(%s,%s)" % (i, k))  # todo ??

    for i in I:
        model.addCons(quicksum(w_[i, k] for k in range(K)) == 1)
        model.addCons(quicksum(a[i, k] * w_[i, k] for k in range(K)) == x[i])
        model.addCons(quicksum(b[i, k] * w_[i, k] for k in range(K)) == c[i])

    model.addCons(quicksum(w[i] * d[i] * x[i] for i in I) <= W)

    model.setObjective(quicksum(c[i] for i in I), "minimize")

    model.data = x, w
    return model

In [79]:
from pyscipopt import SCIP_PARAMSETTING


if __name__ == "__main__":
    # multiple item EOQ
    I, F, h, d, w = multidict(
        {1: [300, 10, 10, 20],
         2: [300, 10, 30, 40],
         3: [300, 10, 50, 10]}
    )
    W = 2000
    K = 1000
    a0, aK = 0.1, 10
    model = eoq(I, F, h, d, w, W, a0, aK, K)
#     model.setPresolve(SCIP_PARAMSETTING.OFF)
#     model.setHeuristics(SCIP_PARAMSETTING.OFF)
#     model.disablePropagation()
#     model.setPresolve(SCIP_PARAMSETTING.OFF)
    model.optimize()

    x, w = model.data
    EPS = 1.e-6
    for v in x:
        if x[v].isInLP():
            print(x[v].name)
        if model.getVal(x[v]) >= EPS:
            print(x[v].name, "=", model.getVal(x[v]))

    print("Optimal value:", model.getObjVal())

x(1) = 1.9315000000000002
x(2) = 0.9442083333333332
x(3) = 0.9613
Optimal value: 1263.6617441394587


In [80]:
is_in_lp = x[2].isInLP()
if is_in_lp:
    print("Variable is in LP!")
    print(f"Variable value in LP is {x.getLPSol()}")
else:
    print("Variable is not in LP!")
# x[1].getCol()

Variable is not in LP!


In [81]:
import gurobipy as gp
from gurobipy import GRB

try:
    # Create a new model
    model = gp.Model("basis_example")

    # Create decision variables with lower bounds of 0
    x = model.addVar(lb=0, name="x")
    y = model.addVar(lb=0, name="y")

    # Set the objective: minimize x + y
    model.setObjective(x + y, GRB.MINIMIZE)

    # Add constraints:
    # Constraint 1: x + 2y >= 10
    model.addConstr(x + 2*y >= 10, name="c0")
    # Constraint 2: 3x + y >= 12
    model.addConstr(3*x + y >= 12, name="c1")

    # Optimize the model
    model.optimize()

    # Retrieve basis statuses after optimization
    var_basis = model.getAttr(GRB.Attr.VBasis, model.getVars())
    constr_basis = model.getAttr(GRB.Attr.CBasis, model.getConstrs())

    # Print the basis status for each variable
    print("Variable basis statuses:")
    for v, basis in zip(model.getVars(), var_basis):
        # basis status codes: 0 = basic, -1 = nonbasic at lower bound,
        # -2 = nonbasic at upper bound, -3 = nonbasic and free
        print(f"  {v.VarName}: {basis}")

    # Print the basis status for each constraint
    print("\nConstraint basis statuses:")
    for c, basis in zip(model.getConstrs(), constr_basis):
        # Similar interpretation applies for constraints
        print(f"  {c.ConstrName}: {basis}")

except gp.GurobiError as e:
    print(f"Gurobi Error {e.errno}: {e}")
except Exception as e:
    print(f"An error occurred: {e}")


Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-1255U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 2 rows, 2 columns and 4 nonzeros
Model fingerprint: 0x2b2674f1
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 1e+01]
Presolve time: 0.03s
Presolved: 2 rows, 2 columns, 4 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   8.000000e+00   0.000000e+00      0s
       2    6.4000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.04 seconds (0.00 work units)
Optimal objective  6.400000000e+00
Variable basis statuses:
  x: 0
  y: 0

Constraint basis statuses:
  c0: -1
  c1: -1
